# Week 1 — Solved

All exercises solved. Data fetched live from SF OpenData API.

## Setup & Data Loading

In [ ]:
import requests, pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker, warnings
warnings.filterwarnings('ignore')

API_NOW  = 'https://data.sfgov.org/resource/wg3w-h783.json'
API_THEN = 'https://data.sfgov.org/resource/tmnf-yvry.json'

def fetch_all(url, params=None, chunk=50_000):
    """Page through Socrata JSON endpoint, return DataFrame."""
    if params is None: params = {}
    records, offset = [], 0
    while True:
        r = requests.get(url, params={**params, '$limit': chunk, '$offset': offset}, timeout=120)
        r.raise_for_status()
        batch = r.json()
        if not batch: break
        records.extend(batch)
        offset += chunk
        if len(batch) < chunk: break
    return pd.DataFrame(records)

print('Fetching 2018-present data …')
df_now_raw = fetch_all(API_NOW)
print(f'  → {len(df_now_raw):,} rows')

## Exercise 2.2 — Load & Clean

In [ ]:
col_map = {'incident_category':'category','incident_date':'date','incident_time':'time',
           'police_district':'district','latitude':'lat','longitude':'lon'}
keep = [c for c in col_map if c in df_now_raw.columns]
df = df_now_raw[keep].rename(columns=col_map).copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['weekday'] = df['date'].dt.day_name()
df['hour']    = pd.to_datetime(df['time'], format='%H:%M', errors='coerce').dt.hour
# Keep only complete years
year_counts = df.groupby('year')['date'].count()
full_years  = year_counts[year_counts > year_counts.median() * 0.9].index
df = df[df['year'].isin(full_years)].copy()
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
print(f'Total incidents : {len(df):,}')
print(f'Date range      : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Incident types  : {df["category"].nunique()}')
df.head(3)

## Exercise 3.1 — Crime categories

In [ ]:
cat_counts = df['category'].value_counts()
print('Most common  :', cat_counts.index[0],  f'({cat_counts.iloc[0]:,})')
print('Least common :', cat_counts.index[-1], f'({cat_counts.iloc[-1]:,})')
print(f'\nAll {len(cat_counts)} categories:')
print(cat_counts.to_string())

## Exercise 3.2 — Bar chart of crime categories

In [ ]:
skip = {'Other Miscellaneous', 'Other Offenses', 'Miscellaneous Investigation'}
plot_counts = cat_counts[~cat_counts.index.isin(skip)].head(30)
fig, ax = plt.subplots(figsize=(10, 9))
colors = plt.cm.viridis_r(np.linspace(0.15, 0.85, len(plot_counts)))
ax.barh(plot_counts.index[::-1], plot_counts.values[::-1], color=colors[::-1])
ax.set_xlabel('Number of incidents', fontsize=12)
ax.set_title('Top 30 Crime Categories in San Francisco (2018–present)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('crime_categories.png', dpi=150, bbox_inches='tight'); plt.show()

## Exercise 4.1 — Yearly crime counts

In [ ]:
yearly = df.groupby('year').size()
print('Most incidents  :', yearly.idxmax(), f'({yearly.max():,})')
print('Fewest incidents:', yearly.idxmin(), f'({yearly.min():,})')
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yearly.index, yearly.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Year'); ax.set_ylabel('Number of incidents')
ax.set_title('Total SF Crime Incidents per Year (2018–present)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
if 2020 in yearly.index:
    ax.annotate('COVID-19\ndrop', xy=(2020, yearly[2020]),
                xytext=(2020.3, yearly[2020]+5000),
                arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('yearly_crime.png', dpi=150, bbox_inches='tight'); plt.show()

## Focus Crimes Definition

The 16 standard focus crimes used throughout the course:

In [ ]:
FOCUS_CRIMES_CANDIDATES = [
    'Assault', 'Burglary', 'Drug Offense', 'Larceny Theft',
    'Motor Vehicle Theft', 'Robbery', 'Vandalism', 'Weapons Offense',
    'Arson', 'Disorderly Conduct', 'Fraud', 'Prostitution',
    'Sex Offense', 'Stolen Property', 'Suspicious Occ', 'Traffic Violation Arrest'
]
available = df['category'].unique()
FOCUS_CRIMES = [c for c in FOCUS_CRIMES_CANDIDATES if c in available]
print(f'Focus crimes found ({len(FOCUS_CRIMES)}):')
for c in FOCUS_CRIMES: print(' ', c)

## Exercise 4.2 — Focus crime trends (4×4 grid)

In [ ]:
df_focus = df[df['category'].isin(FOCUS_CRIMES)].copy()
pivot = df_focus.groupby(['year','category']).size().unstack(fill_value=0)
ncols, n = 4, len(FOCUS_CRIMES)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows*3), sharey=False)
axes = axes.flatten()
for i, crime in enumerate(FOCUS_CRIMES):
    ax = axes[i]
    if crime in pivot.columns:
        s = pivot[crime]
        ax.bar(s.index, s.values, color='steelblue', edgecolor='white')
    ax.set_title(crime, fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(axis='y', alpha=0.3)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig.suptitle('Year-by-Year Trends for Focus Crimes in San Francisco', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('focus_crime_trends.png', dpi=150, bbox_inches='tight'); plt.show()

## Exercise 4.3 — Interpretation

**Three notable patterns:**

1. **COVID-19 drop (2020):** Nearly all crime types show a sharp decline in 2020 due to lockdowns and reduced public activity. Larceny Theft, Assault, and Prostitution fell dramatically. However, *Motor Vehicle Theft* was more resilient — parked cars were still available targets even when streets emptied.

2. **Drug Offense long-term decline:** Drug offenses show a sustained multi-year downward trend, not just a COVID blip. This reflects changing enforcement priorities and prosecutorial decisions in SF — a textbook example of 'dirty data': the data shows *arrests*, not drug use. Policy changes look identical to behavioral changes in the numbers.

3. **Motor Vehicle Theft surge post-COVID:** While most crimes recovered slowly, Motor Vehicle Theft *exceeded* pre-pandemic levels by 2021–2022. This mirrors a national trend linked to widely-shared TikTok videos showing how to steal certain Kia/Hyundai models, plus increased demand for car parts during supply-chain disruptions.